In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
all_cars = []

In [19]:
url = "https://www.pakwheels.com/used-cars/search/-/ct_karachi/"

In [27]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [34]:
response = requests.get(url, headers=headers)

In [40]:
soup = BeautifulSoup(response.text, "html.parser")

In [45]:
ads = soup.select("div.well.search-list.ad-container")

In [52]:
for ad in ads:
    try:
        car_name = ad.select_one("a.car-name h3").text.strip()
        city = ad.select_one("ul.search-vehicle-info li").text.strip()
        price = ad.select_one("div.price-details").text.strip()

        detail_items = [li.text.strip() for li in ad.select("ul.search-vehicle-info-2 li")]
        details = " | ".join(detail_items)

        all_cars.append({
            "Car Name": car_name,
            "City": city,
            "Details": details,
            "Price": price,
        })

    except:
        pass

In [57]:
df = pd.DataFrame(all_cars)

In [74]:
df['Details']

0             2010 | 168,033 km | CNG | 1000 cc | Manual
1        2013 | 79,000 km | Petrol | 3000 cc | Automatic
2      2022 | 4,800 km | Electric | 107.0 kWh | Autom...
3      2022 | 2,500 km | Electric | 93.4 kWh | Automatic
4        2017 | 50,000 km | Petrol | 1600 cc | Automatic
                             ...                        
227      2017 | 87,000 km | Petrol | 1800 cc | Automatic
228        1994 | 400,000 km | Petrol | 1000 cc | Manual
229      2021 | 51,000 km | Petrol | 1300 cc | Automatic
230         2003 | 150,000 km | Petrol | 800 cc | Manual
231      2023 | 60,000 km | Petrol | 1200 cc | Automatic
Name: Details, Length: 232, dtype: object

In [ ]:
# Chech number of columns created through split function
temp_df = df['Details'].str.split('|', expand=True)

In [ ]:
# Assign meaningful column names based on expected order of details (For safe side))
for i in range(temp_df.shape[1]):
    col_names = ['Year', 'Mileage', 'Fuel', 'Engine', 'Transmission', 'Extra']
    if i < len(col_names):
        df[col_names[i]] = temp_df[i].str.strip()

In [86]:
def clean_price(price):
    price = price.replace('PKR', '').replace(',', '').strip()
    if 'lacs' in price:
        return float(price.replace('lacs', '').strip()) * 100000
    elif 'crore' in price:
        return float(price.replace('crore', '').strip()) * 10000000
    return price

df['Price (PKR)'] = df['Price'].apply(clean_price)

In [ ]:
# Remove 'km' aur commas and replace with number
df['Mileage'] = df['Mileage'].str.replace('km', '').str.replace(',', '').str.strip().astype(float)

In [ ]:
# Remove 'cc' aur 'kWh'
df['Engine'] = df['Engine'].str.replace('cc', '').str.replace('kWh', '').str.strip().astype(float)

In [ ]:
# 1. 'for Sale' (and extra spaces) remove
df['Car Name'] = df['Car Name'].str.replace('for Sale', '', case=False).str.strip()

# 2. 4-digit Year remove (use regex)
df['Car Name'] = df['Car Name'].str.replace(r'\d{4}', '', regex=True).str.strip()

# 3. Double spaces into single space (cleaning)
df['Car Name'] = df['Car Name'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
columns_to_remove = ['Details', 'Price', 'Extra']

df.drop(columns=columns_to_remove, inplace=True)

In [96]:
df.head()

,Car Name,City,Year,Mileage,Fuel,Engine,Transmission,Price (PKR)
0,Suzuki Alto VXR (CNG),Karachi,2010,168033.0,CNG,1000.0,Manual,1350000.0
1,Porsche Cayenne Base Model,Karachi,2013,79000.0,Petrol,3000.0,Automatic,17500000.0
2,Mercedes Benz EQS 450 4MATIC,Karachi,2022,4800.0,Electric,107.0,Automatic,40000000.0
3,Audi e-tron GT Standard,Karachi,2022,2500.0,Electric,93.4,Automatic,32500000.0
4,Mercedes Benz C Class C180 AMG,Karachi,2017,50000.0,Petrol,1600.0,Automatic,13500000.0


In [ ]:
# Save into new CSV file
df.to_csv("PakWheels_Karachi_Cleaned.csv", index=False)